In [37]:
# Core Python libraries
import os
import sys
import json
import logging
import sqlite3
import asyncio
from pathlib import Path
from typing import Optional, Literal

# Environment and configuration
from dotenv import load_dotenv

# Pydantic for structured evaluation outputs
from pydantic import BaseModel, Field

# LangChain and Groq
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

# SQL parsing and validation
from sqlglot import parse, exp

# MCP server and client
from mcp.server.fastmcp import FastMCP
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

In [38]:
# Project root directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Main project directories
DATA_DIR = PROJECT_ROOT / "data"
MCP_DIR = PROJECT_ROOT / "mcp_servers"
SCHEMA_SERVER_DIR = MCP_DIR / "schema"
WRITER_SERVER_DIR = MCP_DIR / "writer"
VERIFIER_SERVER_DIR = MCP_DIR / "verifier"
AGENT_DIR = PROJECT_ROOT / "agents"

# Database and existing MCP server paths
DATABASE_PATH = DATA_DIR / "verisql_demo.db"
SCHEMA_SERVER_PATH = SCHEMA_SERVER_DIR / "server.py"
WRITER_SERVER_PATH = WRITER_SERVER_DIR / "server.py"

# Day 3 file paths
EVALUATOR_AGENT_PATH = AGENT_DIR / "evaluator.py"
VERIFIER_SERVER_PATH = VERIFIER_SERVER_DIR / "server.py"

# Environment configuration
load_dotenv(PROJECT_ROOT / ".env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Create directories if they do not exist
VERIFIER_SERVER_DIR.mkdir(parents=True, exist_ok=True)
AGENT_DIR.mkdir(parents=True, exist_ok=True)

# Verify project configuration
print("Project root:", PROJECT_ROOT)
print("Database exists:", DATABASE_PATH.exists())
print("Schema MCP exists:", SCHEMA_SERVER_PATH.exists())
print("Writer MCP exists:", WRITER_SERVER_PATH.exists())
print("Groq API key loaded:", bool(GROQ_API_KEY))
print("Verifier directory:", VERIFIER_SERVER_DIR)
print("Evaluator agent path:", EVALUATOR_AGENT_PATH)

Project root: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL
Database exists: True
Schema MCP exists: True
Writer MCP exists: True
Groq API key loaded: True
Verifier directory: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL\mcp_servers\verifier
Evaluator agent path: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL\agents\evaluator.py


In [39]:
class EvaluationResult(BaseModel):
    syntax_valid: bool = Field(
        description="Whether the SQL syntax is valid"
    )
    schema_valid: bool = Field(
        description="Whether all referenced tables and columns exist"
    )
    safety_valid: bool = Field(
        description="Whether the SQL passes read-only safety checks"
    )
    execution_success: bool = Field(
        description="Whether the SQL executes successfully"
    )
    semantic_correct: bool = Field(
        description="Whether the SQL correctly answers the user's question"
    )
    decision: Literal["accept", "reject"] = Field(
        description="Whether the SQL should be accepted or rejected"
    )
    errors: list[str] = Field(
        description="Specific errors found during evaluation"
    )
    feedback: str = Field(
        description="Actionable feedback for the Writer Agent"
    )
    confidence: float = Field(
        ge=0.0,
        le=1.0,
        description="Evaluator confidence between 0 and 1"
    )

print("EvaluationResult schema created successfully.")

EvaluationResult schema created successfully.


In [40]:
def validate_sql_syntax(sql_query: str) -> dict:
    """Validate SQL syntax and ensure only one query is provided."""

    try:
        parsed_queries = parse(sql_query, read="sqlite")

        if len(parsed_queries) != 1 or parsed_queries[0] is None:
            return {
                "valid": False,
                "error": "Only one SQL statement is allowed."
            }

        query = parsed_queries[0]

        if not isinstance(query, exp.Query):
            return {
                "valid": False,
                "error": "Only SELECT queries are allowed."
            }

        return {
            "valid": True,
            "error": None
        }

    except Exception as error:
        return {
            "valid": False,
            "error": str(error)
        }


def validate_sql_safety(sql_query: str) -> dict:
    """Reject modifying operations and access to sensitive columns."""

    syntax_result = validate_sql_syntax(sql_query)

    if not syntax_result["valid"]:
        return {
            "valid": False,
            "error": syntax_result["error"]
        }

    query = parse(sql_query, read="sqlite")[0]

    restricted_operations = (
        exp.Insert,
        exp.Update,
        exp.Delete,
        exp.Create,
        exp.Drop,
        exp.Alter,
        exp.Command
    )

    for operation in restricted_operations:
        if query.find(operation):
            return {
                "valid": False,
                "error": f"Restricted SQL operation: {operation.__name__}"
            }

    sensitive_keywords = {
        "email", "phone", "password", "token",
        "ssn", "credit_card"
    }

    for column in query.find_all(exp.Column):
        if column.name.lower() in sensitive_keywords:
            return {
                "valid": False,
                "error": f"Access to sensitive column '{column.name}' is restricted."
            }

    return {
        "valid": True,
        "error": None
    }


# Test the validation functions
test_queries = [
    "SELECT employee_name FROM employees;",
    "DELETE FROM employees;",
    "SELECT * FROM employees; DROP TABLE employees;",
    "SELECT email FROM employees;",
    "SELECT employee_name FROM employees WHERE;"
]

for sql in test_queries:
    print("\nSQL:", sql)
    print("Syntax:", validate_sql_syntax(sql))
    print("Safety:", validate_sql_safety(sql))


SQL: SELECT employee_name FROM employees;
Syntax: {'valid': True, 'error': None}
Safety: {'valid': True, 'error': None}

SQL: DELETE FROM employees;
Syntax: {'valid': False, 'error': 'Only SELECT queries are allowed.'}
Safety: {'valid': False, 'error': 'Only SELECT queries are allowed.'}

SQL: SELECT * FROM employees; DROP TABLE employees;
Syntax: {'valid': False, 'error': 'Only one SQL statement is allowed.'}
Safety: {'valid': False, 'error': 'Only one SQL statement is allowed.'}

SQL: SELECT email FROM employees;
Syntax: {'valid': True, 'error': None}
Safety: {'valid': False, 'error': "Access to sensitive column 'email' is restricted."}

SQL: SELECT employee_name FROM employees WHERE;
Syntax: {'valid': False, 'error': "Required keyword: 'this' missing for <class 'sqlglot.expressions.query.Where'>. Line 1, Col: 41.\n  SELECT employee_name FROM employees \x1bWHERE\x1b;"}
Safety: {'valid': False, 'error': "Required keyword: 'this' missing for <class 'sqlglot.expressions.query.Where'>. 

In [41]:
def validate_sql_schema(sql_query: str) -> dict:
    """Validate referenced tables and columns against the database schema."""

    syntax_result = validate_sql_syntax(sql_query)

    if not syntax_result["valid"]:
        return {
            "valid": False,
            "error": syntax_result["error"]
        }

    try:
        query = parse(sql_query, read="sqlite")[0]

        connection = sqlite3.connect(
            f"file:{DATABASE_PATH.resolve().as_posix()}?mode=ro",
            uri=True
        )

        cursor = connection.cursor()

        # SQLite validates table and column references during EXPLAIN.
        cursor.execute("PRAGMA query_only = ON")
        cursor.execute("EXPLAIN " + query.sql(dialect="sqlite"))

        connection.close()

        return {
            "valid": True,
            "error": None
        }

    except sqlite3.Error as error:
        connection.close()

        return {
            "valid": False,
            "error": str(error)
        }


# Test valid and invalid column references
schema_test_queries = [
    "SELECT employee_name FROM employees;",
    "SELECT name FROM employees;",
    "SELECT employee_name FROM unknown_table;"
]

for sql in schema_test_queries:
    print("\nSQL:", sql)
    print("Schema:", validate_sql_schema(sql))


SQL: SELECT employee_name FROM employees;
Schema: {'valid': True, 'error': None}

SQL: SELECT name FROM employees;
Schema: {'valid': False, 'error': 'no such column: name'}

SQL: SELECT employee_name FROM unknown_table;
Schema: {'valid': False, 'error': 'no such table: unknown_table'}


In [42]:
def validate_sql_execution(sql_query: str, max_rows: int = 20) -> dict:
    """Execute a validated SELECT query using a read-only connection."""

    safety_result = validate_sql_safety(sql_query)

    if not safety_result["valid"]:
        return {
            "valid": False,
            "error": safety_result["error"],
            "columns": [],
            "rows": []
        }

    connection = None

    try:
        database_uri = (
            f"file:{DATABASE_PATH.resolve().as_posix()}?mode=ro"
        )

        connection = sqlite3.connect(database_uri, uri=True)
        connection.execute("PRAGMA query_only = ON")

        cursor = connection.cursor()

        # Execute the query with a result limit
        limited_query = f"""
            SELECT *
            FROM ({sql_query.rstrip().rstrip(';')})
            LIMIT {max_rows}
        """

        cursor.execute(limited_query)

        rows = cursor.fetchall()
        columns = [description[0] for description in cursor.description]

        return {
            "valid": True,
            "error": None,
            "columns": columns,
            "rows": rows
        }

    except sqlite3.Error as error:
        return {
            "valid": False,
            "error": str(error),
            "columns": [],
            "rows": []
        }

    finally:
        if connection is not None:
            connection.close()


# Test SQL execution
execution_test_queries = [
    "SELECT employee_name, salary FROM employees;",
    "SELECT employee_name FROM employees WHERE salary > 100000;",
    "SELECT name FROM employees;"
]

for sql in execution_test_queries:
    print("\nSQL:", sql)
    print(json.dumps(
        validate_sql_execution(sql),
        indent=4,
        default=str
    ))


SQL: SELECT employee_name, salary FROM employees;
{
    "valid": true,
    "error": null,
    "columns": [
        "employee_name",
        "salary"
    ],
    "rows": [
        [
            "Arun",
            75000.0
        ],
        [
            "Priya",
            82000.0
        ],
        [
            "Rahul",
            68000.0
        ],
        [
            "Sneha",
            90000.0
        ],
        [
            "Kiran",
            78000.0
        ],
        [
            "Arun",
            85000.0
        ],
        [
            "Priya",
            92000.0
        ],
        [
            "Rahul",
            75000.0
        ],
        [
            "Sneha",
            68000.0
        ],
        [
            "Kiran",
            60000.0
        ]
    ]
}

SQL: SELECT employee_name FROM employees WHERE salary > 100000;
{
    "valid": true,
    "error": null,
    "columns": [
        "employee_name"
    ],
    "rows": []
}

SQL: SELECT name FROM employees;


In [43]:
verifier_server_code = '''
from pathlib import Path
import sys

# Add project root to Python path
PROJECT_ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

from mcp.server.fastmcp import FastMCP
from agents.evaluator import (
    validate_sql_syntax,
    validate_sql_safety,
    validate_sql_schema,
    validate_sql_execution
)

mcp_server = FastMCP("VeriSQL-Verifier")


@mcp_server.tool()
def check_sql_syntax(sql_query: str) -> dict:
    """Check SQL syntax and statement count."""
    return validate_sql_syntax(sql_query)


@mcp_server.tool()
def check_sql_safety(sql_query: str) -> dict:
    """Check read-only restrictions and sensitive column access."""
    return validate_sql_safety(sql_query)


@mcp_server.tool()
def check_sql_schema(sql_query: str) -> dict:
    """Check referenced database tables and columns."""
    return validate_sql_schema(sql_query)


@mcp_server.tool()
def execute_sql_readonly(sql_query: str) -> dict:
    """Execute a permitted SQL query using a read-only connection."""
    return validate_sql_execution(sql_query)


if __name__ == "__main__":
    mcp_server.run(transport="stdio")
'''

VERIFIER_SERVER_PATH.write_text(
    verifier_server_code,
    encoding="utf-8"
)

print("Verifier MCP server file created:", VERIFIER_SERVER_PATH)

Verifier MCP server file created: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL\mcp_servers\verifier\server.py


In [44]:
evaluator_module_code = '''
import sqlite3
from pathlib import Path

from sqlglot import parse, exp


PROJECT_ROOT = Path(__file__).resolve().parents[1]
DATABASE_PATH = PROJECT_ROOT / "data" / "verisql_demo.db"


def validate_sql_syntax(sql_query: str) -> dict:
    """Validate SQL syntax and ensure exactly one query is provided."""

    try:
        parsed_queries = parse(sql_query, read="sqlite")

        if len(parsed_queries) != 1 or parsed_queries[0] is None:
            return {"valid": False, "error": "Only one SQL statement is allowed."}

        if not isinstance(parsed_queries[0], exp.Query):
            return {"valid": False, "error": "Only SELECT queries are allowed."}

        return {"valid": True, "error": None}

    except Exception as error:
        return {"valid": False, "error": str(error)}


def validate_sql_safety(sql_query: str) -> dict:
    """Reject modifying operations and access to sensitive columns."""

    syntax_result = validate_sql_syntax(sql_query)

    if not syntax_result["valid"]:
        return syntax_result

    query = parse(sql_query, read="sqlite")[0]

    restricted_operations = (
        exp.Insert, exp.Update, exp.Delete,
        exp.Create, exp.Drop, exp.Alter, exp.Command
    )

    for operation in restricted_operations:
        if query.find(operation):
            return {
                "valid": False,
                "error": f"Restricted SQL operation: {operation.__name__}"
            }

    sensitive_keywords = {
        "email", "phone", "password", "token",
        "ssn", "credit_card"
    }

    for column in query.find_all(exp.Column):
        if column.name.lower() in sensitive_keywords:
            return {
                "valid": False,
                "error": f"Access to sensitive column '{column.name}' is restricted."
            }

    return {"valid": True, "error": None}


def validate_sql_schema(sql_query: str) -> dict:
    """Check referenced tables and columns against the SQLite database."""

    syntax_result = validate_sql_syntax(sql_query)

    if not syntax_result["valid"]:
        return syntax_result

    connection = None

    try:
        query = parse(sql_query, read="sqlite")[0]

        database_uri = (
            f"file:{DATABASE_PATH.resolve().as_posix()}?mode=ro"
        )

        connection = sqlite3.connect(database_uri, uri=True)
        connection.execute("PRAGMA query_only = ON")
        connection.execute(
            "EXPLAIN " + query.sql(dialect="sqlite")
        )

        return {"valid": True, "error": None}

    except sqlite3.Error as error:
        return {"valid": False, "error": str(error)}

    finally:
        if connection is not None:
            connection.close()


def validate_sql_execution(sql_query: str, max_rows: int = 20) -> dict:
    """Execute SQL using a read-only connection and return limited results."""

    safety_result = validate_sql_safety(sql_query)

    if not safety_result["valid"]:
        return {
            "valid": False,
            "error": safety_result["error"],
            "columns": [],
            "rows": []
        }

    connection = None

    try:
        database_uri = (
            f"file:{DATABASE_PATH.resolve().as_posix()}?mode=ro"
        )

        connection = sqlite3.connect(database_uri, uri=True)
        connection.execute("PRAGMA query_only = ON")

        query = parse(sql_query, read="sqlite")[0]
        clean_sql = query.sql(dialect="sqlite")

        cursor = connection.cursor()
        cursor.execute(
            f"SELECT * FROM ({clean_sql}) LIMIT ?",
            (max_rows,)
        )

        rows = cursor.fetchall()
        columns = [item[0] for item in cursor.description]

        return {
            "valid": True,
            "error": None,
            "columns": columns,
            "rows": rows
        }

    except sqlite3.Error as error:
        return {
            "valid": False,
            "error": str(error),
            "columns": [],
            "rows": []
        }

    finally:
        if connection is not None:
            connection.close()
'''

EVALUATOR_AGENT_PATH.write_text(
    evaluator_module_code,
    encoding="utf-8"
)

print("Evaluator module created:", EVALUATOR_AGENT_PATH)
print("File size:", EVALUATOR_AGENT_PATH.stat().st_size, "bytes")

Evaluator module created: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL\agents\evaluator.py
File size: 4219 bytes


In [45]:
from agents.evaluator import (validate_sql_syntax as standalone_syntax,
    validate_sql_safety as standalone_safety,
    validate_sql_schema as standalone_schema,
    validate_sql_execution as standalone_execution)

test_sql = """
SELECT e.employee_name, d.department_name
FROM employees e
JOIN departments d
ON e.department_id = d.department_id
WHERE d.department_name = 'Engineering';
"""

print("SYNTAX:", standalone_syntax(test_sql))
print("SAFETY:", standalone_safety(test_sql))
print("SCHEMA:", standalone_schema(test_sql))

execution_result = standalone_execution(test_sql)

print("\nEXECUTION VALID:", execution_result["valid"])
print("COLUMNS:", execution_result["columns"])
print("ROWS:", execution_result["rows"])

SYNTAX: {'valid': True, 'error': None}
SAFETY: {'valid': True, 'error': None}
SCHEMA: {'valid': True, 'error': None}

EXECUTION VALID: True
COLUMNS: ['employee_name', 'department_name']
ROWS: [('Arun', 'Engineering'), ('Rahul', 'Engineering'), ('Arun', 'Engineering'), ('Priya', 'Engineering')]


In [46]:
async def test_verifier_tool_discovery():
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(VERIFIER_SERVER_PATH)]
    )

    with open(
        PROJECT_ROOT / "verifier_mcp_errors.log",
        "w",
        encoding="utf-8"
    ) as server_log:

        async with stdio_client(
            server_params,
            errlog=server_log
        ) as (read, write):

            async with ClientSession(read, write) as session:
                await session.initialize()

                tools = await session.list_tools()

                print("Available Verifier MCP tools:\n")

                for tool in tools.tools:
                    print(f"- {tool.name}: {tool.description}")


await test_verifier_tool_discovery()

Available Verifier MCP tools:

- check_sql_syntax: Check SQL syntax and statement count.
- check_sql_safety: Check read-only restrictions and sensitive column access.
- check_sql_schema: Check referenced database tables and columns.
- execute_sql_readonly: Execute a permitted SQL query using a read-only connection.


In [47]:
async def test_verifier_mcp_tools():
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(VERIFIER_SERVER_PATH)]
    )

    test_sql = """
    SELECT e.employee_name, d.department_name
    FROM employees e
    JOIN departments d
    ON e.department_id = d.department_id
    WHERE d.department_name = 'Engineering';
    """

    with open(
        PROJECT_ROOT / "verifier_mcp_errors.log",
        "w",
        encoding="utf-8"
    ) as server_log:

        async with stdio_client(
            server_params,
            errlog=server_log
        ) as (read, write):

            async with ClientSession(read, write) as session:
                await session.initialize()

                syntax = await session.call_tool(
                    "check_sql_syntax",
                    {"sql_query": test_sql}
                )

                safety = await session.call_tool(
                    "check_sql_safety",
                    {"sql_query": test_sql}
                )

                schema = await session.call_tool(
                    "check_sql_schema",
                    {"sql_query": test_sql}
                )

                execution = await session.call_tool(
                    "execute_sql_readonly",
                    {"sql_query": test_sql}
                )

                print("SYNTAX RESULT:")
                print(syntax.content[0].text)

                print("\nSAFETY RESULT:")
                print(safety.content[0].text)

                print("\nSCHEMA RESULT:")
                print(schema.content[0].text)

                print("\nEXECUTION RESULT:")
                print(execution.content[0].text)


await test_verifier_mcp_tools()

SYNTAX RESULT:
{
  "valid": true,
  "error": null
}

SAFETY RESULT:
{
  "valid": true,
  "error": null
}

SCHEMA RESULT:
{
  "valid": true,
  "error": null
}

EXECUTION RESULT:
{
  "valid": true,
  "error": null,
  "columns": [
    "employee_name",
    "department_name"
  ],
  "rows": [
    [
      "Arun",
      "Engineering"
    ],
    [
      "Rahul",
      "Engineering"
    ],
    [
      "Arun",
      "Engineering"
    ],
    [
      "Priya",
      "Engineering"
    ]
  ]
}


In [48]:
# Initialise the Evaluator LLM
evaluator_llm = ChatGroq(api_key=GROQ_API_KEY,
    model="openai/gpt-oss-20b",
    temperature=0)

# Structured output parser
evaluation_parser = PydanticOutputParser(pydantic_object=EvaluationResult)

# Evaluator system prompt
evaluator_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are an independent SQL Evaluation Agent in VeriSQL.

        Your responsibility is to evaluate whether generated SQL
        correctly answers the user's natural language question.

        Analyse the following carefully:

        1. Whether the SQL is syntactically valid.
        2. Whether referenced tables and columns are correct.
        3. Whether the query follows read-only safety requirements.
        4. Whether the query executes successfully.
        5. Whether the SQL logically answers the user's question.

        Pay special attention to:
        - Missing JOIN operations.
        - Incorrect JOIN conditions.
        - Missing WHERE conditions.
        - Incorrect aggregation or GROUP BY.
        - Incorrect filtering.
        - Unnecessary columns.
        - Incorrect interpretation of the question.

        IMPORTANT:
        - Do not assume that successful execution means correctness.
        - Do not invent database schema or query results.
        - Use the supplied validation results as evidence.
        - If semantic correctness cannot be established, reject the query.
        - Provide specific and actionable feedback for the Writer Agent.
        - Never approve SQL that fails a deterministic validation check.

        Return your evaluation in the required structured format.

        {format_instructions}
        """
    ),
    (
        "human",
        """
        USER QUESTION:
        {question}

        GENERATED SQL:
        {sql_query}

        VERIFIER VALIDATION REPORT:
        {validation_report}

        SQL EXECUTION RESULTS:
        {execution_results}

        Evaluate whether the generated SQL correctly answers the
        user's question.
        """
    )
])

# Build the Evaluator Agent chain
evaluator_chain = evaluator_prompt | evaluator_llm | evaluation_parser

print("Evaluator LLM initialised.")
print("Structured evaluation parser configured.")
print("Evaluator prompt created.")
print("Evaluator Agent chain created.")

Evaluator LLM initialised.
Structured evaluation parser configured.
Evaluator prompt created.
Evaluator Agent chain created.


In [49]:
async def evaluate_sql(question: str, sql_query: str) -> EvaluationResult:
    """Evaluate generated SQL using Verifier MCP and the Evaluator LLM."""

    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(VERIFIER_SERVER_PATH)]
    )

    with open(
        PROJECT_ROOT / "verifier_mcp_errors.log",
        "w",
        encoding="utf-8"
    ) as server_log:

        async with stdio_client(
            server_params,
            errlog=server_log
        ) as (read, write):

            async with ClientSession(read, write) as session:
                await session.initialize()

                # Run deterministic validation tools
                syntax = await session.call_tool(
                    "check_sql_syntax",
                    {"sql_query": sql_query}
                )

                safety = await session.call_tool(
                    "check_sql_safety",
                    {"sql_query": sql_query}
                )

                schema = await session.call_tool(
                    "check_sql_schema",
                    {"sql_query": sql_query}
                )

                execution = await session.call_tool(
                    "execute_sql_readonly",
                    {"sql_query": sql_query}
                )

                # Convert MCP responses into Python dictionaries
                syntax_result = json.loads(syntax.content[0].text)
                safety_result = json.loads(safety.content[0].text)
                schema_result = json.loads(schema.content[0].text)
                execution_result = json.loads(execution.content[0].text)

    # Combine deterministic validation results
    validation_report = {
        "syntax": syntax_result,
        "safety": safety_result,
        "schema": schema_result,
        "execution": {
            "valid": execution_result["valid"],
            "error": execution_result["error"]
        }
    }

    execution_success = execution_result["valid"]

    # Collect deterministic validation failures
    errors = []

    for check_name, result in [
        ("Syntax", syntax_result),
        ("Safety", safety_result),
        ("Schema", schema_result),
        ("Execution", execution_result)
    ]:
        if not result["valid"]:
            errors.append(f"{check_name}: {result['error']}")

    # Never allow the LLM to override a failed deterministic check
    if errors:
        return EvaluationResult(
            syntax_valid=syntax_result["valid"],
            schema_valid=schema_result["valid"],
            safety_valid=safety_result["valid"],
            execution_success=execution_success,
            semantic_correct=False,
            decision="reject",
            errors=errors,
            feedback="Fix the deterministic validation errors before resubmitting the SQL.",
            confidence=1.0
        )

    # Ask the LLM to evaluate semantic correctness
    evaluation = await evaluator_chain.ainvoke({
        "question": question,
        "sql_query": sql_query,
        "validation_report": json.dumps(validation_report, indent=2),
        "execution_results": json.dumps(
            execution_result,
            indent=2,
            default=str
        ),
        "format_instructions": evaluation_parser.get_format_instructions()
    })

    return evaluation

In [50]:
question = "Find the names of employees working in the Engineering department."

incorrect_sql = """
SELECT employee_name
FROM employees;
"""

incorrect_evaluation = await evaluate_sql(question, incorrect_sql)

print(incorrect_evaluation.model_dump_json(indent=4))

{
    "syntax_valid": true,
    "schema_valid": true,
    "safety_valid": true,
    "execution_success": true,
    "semantic_correct": false,
    "decision": "reject",
    "errors": [
        "The query does not filter employees by the Engineering department; it returns all employees."
    ],
    "feedback": "Add a WHERE clause that restricts the results to employees whose department is Engineering. For example, if the employees table has a column named department_name, use:\n\nSELECT employee_name\nFROM employees\nWHERE department_name = 'Engineering';\n\nIf the department is referenced via a foreign key, join the departments table accordingly: \n\nSELECT e.employee_name\nFROM employees e\nJOIN departments d ON e.department_id = d.id\nWHERE d.department_name = 'Engineering';",
    "confidence": 0.95
}


In [51]:
unsafe_sql = """
SELECT employee_name, email
FROM employees;
"""

unsafe_evaluation = await evaluate_sql(
    "Find employee names and email addresses.",
    unsafe_sql
)

print(unsafe_evaluation.model_dump_json(indent=4))

{
    "syntax_valid": true,
    "schema_valid": true,
    "safety_valid": false,
    "execution_success": false,
    "semantic_correct": false,
    "decision": "reject",
    "errors": [
        "Safety: Access to sensitive column 'email' is restricted.",
        "Execution: Access to sensitive column 'email' is restricted."
    ],
    "feedback": "Fix the deterministic validation errors before resubmitting the SQL.",
    "confidence": 1.0
}


In [52]:
from agents.writer import repair_sql

async def evaluate_and_repair(question: str, initial_sql: str) -> dict:
    """Evaluate generated SQL and attempt one repair if rejected."""

    print("STEP 1: Evaluating initial SQL...")

    initial_evaluation = await evaluate_sql(question, initial_sql)

    print("Initial decision:", initial_evaluation.decision)

    if initial_evaluation.decision == "accept":
        return {
            "initial_sql": initial_sql,
            "initial_evaluation": initial_evaluation.model_dump(),
            "repaired_sql": None,
            "final_evaluation": initial_evaluation.model_dump(),
            "repair_attempted": False
        }

    print("\nSTEP 2: Sending feedback to Writer Agent...")

    repaired_sql_result = await repair_sql(
        question=question,
        previous_sql=initial_sql,
        evaluator_feedback=initial_evaluation.feedback
    )

    repaired_sql = (
        repaired_sql_result.sql_query
        if hasattr(repaired_sql_result, "sql_query")
        else repaired_sql_result["sql_query"]
    )

    print("\nRepaired SQL:")
    print(repaired_sql)

    print("\nSTEP 3: Evaluating repaired SQL...")

    final_evaluation = await evaluate_sql(question, repaired_sql)

    print("Final decision:", final_evaluation.decision)

    return {
        "initial_sql": initial_sql,
        "initial_evaluation": initial_evaluation.model_dump(),
        "repaired_sql": repaired_sql,
        "final_evaluation": final_evaluation.model_dump(),
        "repair_attempted": True
    }

In [53]:
question = "Find the names of employees working in the Engineering department."

initial_sql = """
SELECT employee_name
FROM employees;
"""

repair_result = await evaluate_and_repair(question=question,initial_sql=initial_sql)

print("\nFINAL WORKFLOW REPORT")
print(json.dumps(repair_result, indent=4, default=str))

STEP 1: Evaluating initial SQL...
Initial decision: reject

STEP 2: Sending feedback to Writer Agent...

Repaired SQL:
SELECT e.employee_name
FROM employees e
JOIN departments d ON e.department_id = d.department_id
WHERE d.department_name = 'Engineering';

STEP 3: Evaluating repaired SQL...
Final decision: accept

FINAL WORKFLOW REPORT
{
    "initial_sql": "\nSELECT employee_name\nFROM employees;\n",
    "initial_evaluation": {
        "syntax_valid": true,
        "schema_valid": true,
        "safety_valid": true,
        "execution_success": true,
        "semantic_correct": false,
        "decision": "reject",
        "errors": [
            "The query does not filter employees by the Engineering department; it returns all employees."
        ],
        "feedback": "Add a WHERE clause that restricts the results to employees whose department is Engineering. For example, if the employees table has a column named department_name, use:\n\nSELECT employee_name\nFROM employees\nWHERE dep

In [54]:
async def get_evaluator_schema_context(db_id="verisql_demo"):
    """Retrieve the actual database schema through Schema MCP."""

    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(SCHEMA_SERVER_PATH)]
    )

    with open(
        PROJECT_ROOT / "schema_mcp_errors.log",
        "w",
        encoding="utf-8"
    ) as server_log:

        async with stdio_client(
            server_params,
            errlog=server_log
        ) as (read, write):

            async with ClientSession(read, write) as session:
                await session.initialize()

                schema_result = await session.call_tool(
                    "get_schema",
                    {"db_id": db_id}
                )

                return schema_result.content[0].text


# Retrieve actual database schema
evaluator_schema = await get_evaluator_schema_context()

print("DATABASE SCHEMA CONTEXT:")
print(evaluator_schema)

DATABASE SCHEMA CONTEXT:
{
  "departments": {
    "columns": [
      {
        "name": "department_id",
        "type": "INTEGER",
        "primary_key": true,
        "nullable": true,
        "sensitive": false
      },
      {
        "name": "department_name",
        "type": "TEXT",
        "primary_key": false,
        "nullable": false,
        "sensitive": false
      },
      {
        "name": "location",
        "type": "TEXT",
        "primary_key": false,
        "nullable": true,
        "sensitive": false
      }
    ],
    "foreign_keys": []
  },
  "employee_projects": {
    "columns": [
      {
        "name": "employee_id",
        "type": "INTEGER",
        "primary_key": true,
        "nullable": true,
        "sensitive": false
      },
      {
        "name": "project_id",
        "type": "INTEGER",
        "primary_key": true,
        "nullable": true,
        "sensitive": false
      },
      {
        "name": "assigned_date",
        "type": "TEXT",
        "pri

In [55]:
evaluator_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are an independent SQL Evaluation Agent in VeriSQL.

        Your task is to determine whether generated SQL correctly
        answers the user's natural language question.

        ACTUAL DATABASE SCHEMA:
        {database_schema}

        Follow these evaluation rules:

        1. Check whether the SQL correctly answers the user's question.
        2. Identify missing filters, joins, aggregations and conditions.
        3. Use only tables and columns present in the supplied schema.
        4. Follow the actual foreign-key relationships when suggesting JOINs.
        5. Never invent table names, column names or relationships.
        6. Do not confuse successful execution with semantic correctness.
        7. Treat the Verifier's deterministic results as authoritative.
        8. Reject SQL if any deterministic validation fails.
        9. If the query is incorrect, provide specific repair instructions.
        10. Do not approve SQL when semantic correctness is uncertain.

        IMPORTANT:
        The database schema is authoritative.
        Do not use assumptions that contradict it.

        Return the evaluation in the required structured format.

        {format_instructions}
        """
    ),
    (
        "human",
        """
        USER QUESTION:
        {question}

        GENERATED SQL:
        {sql_query}

        VERIFIER VALIDATION REPORT:
        {validation_report}

        SQL EXECUTION RESULTS:
        {execution_results}

        Evaluate whether the SQL correctly answers the question.
        """
    )
])

evaluator_chain = evaluator_prompt | evaluator_llm | evaluation_parser

print("Schema-aware Evaluator prompt configured.")
print("Actual database schema will be supplied during evaluation.")

Schema-aware Evaluator prompt configured.
Actual database schema will be supplied during evaluation.


In [56]:
async def evaluate_sql(question: str, sql_query: str) -> EvaluationResult:
    """Evaluate SQL using Verifier MCP and schema-aware LLM reasoning."""

    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(VERIFIER_SERVER_PATH)]
    )

    with open(
        PROJECT_ROOT / "verifier_mcp_errors.log",
        "w",
        encoding="utf-8"
    ) as server_log:

        async with stdio_client(
            server_params,
            errlog=server_log
        ) as (read, write):

            async with ClientSession(read, write) as session:
                await session.initialize()

                # Run deterministic validation tools
                tool_results = {}

                for tool_name in [
                    "check_sql_syntax",
                    "check_sql_safety",
                    "check_sql_schema",
                    "execute_sql_readonly"
                ]:
                    result = await session.call_tool(
                        tool_name,
                        {"sql_query": sql_query}
                    )

                    tool_results[tool_name] = json.loads(
                        result.content[0].text
                    )

    syntax_result = tool_results["check_sql_syntax"]
    safety_result = tool_results["check_sql_safety"]
    schema_result = tool_results["check_sql_schema"]
    execution_result = tool_results["execute_sql_readonly"]

    validation_report = {
        "syntax": syntax_result,
        "safety": safety_result,
        "schema": schema_result,
        "execution": {
            "valid": execution_result["valid"],
            "error": execution_result["error"]
        }
    }

    # Collect deterministic failures
    errors = []

    for check_name, result in [
        ("Syntax", syntax_result),
        ("Safety", safety_result),
        ("Schema", schema_result),
        ("Execution", execution_result)
    ]:
        if not result["valid"]:
            errors.append(f"{check_name}: {result['error']}")

    # Reject immediately if any deterministic check fails
    if errors:
        return EvaluationResult(
            syntax_valid=syntax_result["valid"],
            schema_valid=schema_result["valid"],
            safety_valid=safety_result["valid"],
            execution_success=execution_result["valid"],
            semantic_correct=False,
            decision="reject",
            errors=errors,
            feedback="Fix the deterministic validation errors before resubmitting the SQL.",
            confidence=1.0
        )

    # Run schema-aware semantic evaluation
    evaluation = await evaluator_chain.ainvoke({
        "question": question,
        "sql_query": sql_query,
        "database_schema": evaluator_schema,
        "validation_report": json.dumps(
            validation_report,
            indent=2
        ),
        "execution_results": json.dumps(
            execution_result,
            indent=2,
            default=str
        ),
        "format_instructions": evaluation_parser.get_format_instructions()
    })

    return evaluation


print("Schema-aware evaluate_sql function updated.")

Schema-aware evaluate_sql function updated.


In [57]:
question = "Find the names of employees working in the Engineering department."

incorrect_sql = """
SELECT employee_name
FROM employees;
"""

schema_aware_evaluation = await evaluate_sql(
    question,
    incorrect_sql
)

print(schema_aware_evaluation.model_dump_json(indent=4))

{
    "syntax_valid": true,
    "schema_valid": true,
    "safety_valid": true,
    "execution_success": true,
    "semantic_correct": false,
    "decision": "reject",
    "errors": [
        "The query does not filter employees by the Engineering department; it returns all employees."
    ],
    "feedback": "Add a join to the departments table and filter on department_name = 'Engineering'. For example:\n\nSELECT e.employee_name\nFROM employees e\nJOIN departments d ON e.department_id = d.department_id\nWHERE d.department_name = 'Engineering';",
    "confidence": 0.95
}


In [58]:
async def evaluate_and_repair(
    question: str,
    initial_sql: str,
    max_attempts: int = 3
) -> dict:
    """Run controlled Writer-Evaluator repair iterations."""

    current_sql = initial_sql
    history = []

    for attempt in range(1, max_attempts + 1):

        print(f"\nATTEMPT {attempt}: Evaluating SQL...")

        evaluation = await evaluate_sql(question, current_sql)

        history.append({
            "attempt": attempt,
            "sql": current_sql,
            "evaluation": evaluation.model_dump()
        })

        print("Decision:", evaluation.decision)

        if evaluation.decision == "accept":
            print("SQL accepted. Repair loop completed.")
            break

        if attempt == max_attempts:
            print("Maximum repair attempts reached.")
            break

        print("Sending feedback to Writer Agent...")

        repair_result = await repair_sql(
            question=question,
            previous_sql=current_sql,
            evaluator_feedback=evaluation.feedback
        )

        current_sql = (
            repair_result.sql_query
            if hasattr(repair_result, "sql_query")
            else repair_result["sql_query"]
        )

    final_evaluation = history[-1]["evaluation"]

    return {
        "question": question,
        "initial_sql": initial_sql,
        "final_sql": current_sql,
        "final_decision": final_evaluation["decision"],
        "repair_attempts": len(history) - 1,
        "total_evaluations": len(history),
        "history": history
    }


print("Controlled repair workflow created.")
print("Maximum evaluation attempts: 3")

Controlled repair workflow created.
Maximum evaluation attempts: 3


In [59]:
question = "Find the names of employees working in the Engineering department."

initial_sql = """
SELECT employee_name
FROM employees;
"""

retry_result = await evaluate_and_repair(
    question=question,
    initial_sql=initial_sql,
    max_attempts=3
)

print("\nFINAL REPAIR REPORT")
print(json.dumps(retry_result, indent=4, default=str))


ATTEMPT 1: Evaluating SQL...
Decision: reject
Sending feedback to Writer Agent...

ATTEMPT 2: Evaluating SQL...
Decision: accept
SQL accepted. Repair loop completed.

FINAL REPAIR REPORT
{
    "question": "Find the names of employees working in the Engineering department.",
    "initial_sql": "\nSELECT employee_name\nFROM employees;\n",
    "final_sql": "SELECT e.employee_name\nFROM employees e\nJOIN departments d ON e.department_id = d.department_id\nWHERE d.department_name = 'Engineering';",
    "final_decision": "accept",
    "repair_attempts": 1,
    "total_evaluations": 2,
    "history": [
        {
            "attempt": 1,
            "sql": "\nSELECT employee_name\nFROM employees;\n",
            "evaluation": {
                "syntax_valid": true,
                "schema_valid": true,
                "safety_valid": true,
                "execution_success": true,
                "semantic_correct": false,
                "decision": "reject",
                "errors": [
   

In [60]:
EVALUATION_LOG_PATH = DATA_DIR / "evaluation_logs.jsonl"


def log_evaluation(result: dict) -> None:
    """Append a complete evaluation workflow to the JSONL log."""

    EVALUATION_LOG_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        EVALUATION_LOG_PATH,
        "a",
        encoding="utf-8"
    ) as log_file:

        log_file.write(
            json.dumps(result, default=str) + "\n"
        )


print("Evaluation log path:", EVALUATION_LOG_PATH)
print("Evaluation logger created.")

Evaluation log path: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL\data\evaluation_logs.jsonl
Evaluation logger created.


In [61]:
async def evaluate_and_repair(
    question: str,
    initial_sql: str,
    max_attempts: int = 3
) -> dict:
    """Evaluate, repair and persist the complete SQL workflow."""

    current_sql = initial_sql
    history = []

    for attempt in range(1, max_attempts + 1):

        print(f"\nATTEMPT {attempt}: Evaluating SQL...")

        evaluation = await evaluate_sql(question, current_sql)

        history.append({
            "attempt": attempt,
            "sql": current_sql,
            "evaluation": evaluation.model_dump()
        })

        print("Decision:", evaluation.decision)

        if evaluation.decision == "accept":
            print("SQL accepted.")
            break

        if attempt == max_attempts:
            print("Maximum repair attempts reached.")
            break

        print("Sending feedback to Writer Agent...")

        repair_result = await repair_sql(
            question=question,
            previous_sql=current_sql,
            evaluator_feedback=evaluation.feedback
        )

        current_sql = (
            repair_result.sql_query
            if hasattr(repair_result, "sql_query")
            else repair_result["sql_query"]
        )

    final_evaluation = history[-1]["evaluation"]

    workflow_result = {
        "question": question,
        "initial_sql": initial_sql,
        "final_sql": current_sql,
        "final_decision": final_evaluation["decision"],
        "repair_attempts": len(history) - 1,
        "total_evaluations": len(history),
        "history": history
    }

    # Persist the complete workflow
    log_evaluation(workflow_result)

    print("\nEvaluation workflow saved successfully.")

    return workflow_result


print("Repair workflow updated with persistent logging.")

Repair workflow updated with persistent logging.


In [62]:
question = "Find the names of employees working in the Engineering department."

initial_sql = """
SELECT employee_name
FROM employees;
"""

logged_result = await evaluate_and_repair(
    question=question,
    initial_sql=initial_sql,
    max_attempts=3
)

print("\nFinal decision:", logged_result["final_decision"])
print("Repair attempts:", logged_result["repair_attempts"])
print("Total evaluations:", logged_result["total_evaluations"])
print("Log file exists:", EVALUATION_LOG_PATH.exists())


ATTEMPT 1: Evaluating SQL...
Decision: reject
Sending feedback to Writer Agent...

ATTEMPT 2: Evaluating SQL...
Decision: accept
SQL accepted.

Evaluation workflow saved successfully.

Final decision: accept
Repair attempts: 1
Total evaluations: 2
Log file exists: True


In [63]:
with open(EVALUATION_LOG_PATH, "r", encoding="utf-8") as log_file:
    evaluation_records = [
        json.loads(line)
        for line in log_file
        if line.strip()
    ]

print("Total logged workflows:", len(evaluation_records))

latest_record = evaluation_records[-1]

print("\nLatest logged workflow:")
print(json.dumps(latest_record, indent=4))

Total logged workflows: 6

Latest logged workflow:
{
    "question": "Find the names of employees working in the Engineering department.",
    "initial_sql": "\nSELECT employee_name\nFROM employees;\n",
    "final_sql": "SELECT e.employee_name\nFROM employees e\nJOIN departments d ON e.department_id = d.department_id\nWHERE d.department_name = 'Engineering';",
    "final_decision": "accept",
    "repair_attempts": 1,
    "total_evaluations": 2,
    "history": [
        {
            "attempt": 1,
            "sql": "\nSELECT employee_name\nFROM employees;\n",
            "evaluation": {
                "syntax_valid": true,
                "schema_valid": true,
                "safety_valid": true,
                "execution_success": true,
                "semantic_correct": false,
                "decision": "reject",
                "errors": [
                    "The query does not filter employees by the Engineering department; it returns all employees."
                ],
      

In [64]:
def calculate_evaluation_metrics(log_path):
    """Calculate performance metrics from evaluation logs."""

    with open(log_path, "r", encoding="utf-8") as log_file:
        records = [
            json.loads(line)
            for line in log_file
            if line.strip()
        ]

    total = len(records)

    if total == 0:
        return {"error": "No evaluation records found."}

    accepted = sum(
        record["final_decision"] == "accept"
        for record in records
    )

    repaired = [
        record for record in records
        if record["repair_attempts"] > 0
    ]

    initial_rejected = sum(
        record["history"][0]["evaluation"]["decision"] == "reject"
        for record in records
    )

    successful_repairs = sum(
        record["final_decision"] == "accept"
        for record in repaired
    )

    return {
        "total_workflows": total,
        "acceptance_rate": round(accepted / total * 100, 2),
        "initial_rejection_rate": round(initial_rejected / total * 100, 2),
        "repair_success_rate": (
            round(successful_repairs / len(repaired) * 100, 2)
            if repaired else None
        ),
        "average_repair_attempts": round(
            sum(r["repair_attempts"] for r in records) / total, 2
        ),
        "average_evaluations": round(
            sum(r["total_evaluations"] for r in records) / total, 2
        )
    }


evaluation_metrics = calculate_evaluation_metrics(
    EVALUATION_LOG_PATH
)

print(json.dumps(evaluation_metrics, indent=4))

{
    "total_workflows": 6,
    "acceptance_rate": 100.0,
    "initial_rejection_rate": 33.33,
    "repair_success_rate": 100.0,
    "average_repair_attempts": 0.33,
    "average_evaluations": 1.33
}


In [65]:
evaluation_test_cases = [
    {
        "id": "TC01",
        "question": "Find all employee names.",
        "error_type": "simple_select"
    },
    {
        "id": "TC02",
        "question": "Find employees earning more than 80000.",
        "error_type": "filtering"
    },
    {
        "id": "TC03",
        "question": "Find employees working in Engineering.",
        "error_type": "join"
    },
    {
        "id": "TC04",
        "question": "Find the average salary in each department.",
        "error_type": "aggregation"
    },
    {
        "id": "TC05",
        "question": "Find employee names using their name column.",
        "error_type": "invalid_column"
    },
    {
        "id": "TC06",
        "question": "Find employees assigned to the AI Platform project.",
        "error_type": "missing_join"
    },
    {
        "id": "TC07",
        "question": "Find employees working in Analytics.",
        "error_type": "incorrect_filter"
    },
    {
        "id": "TC08",
        "question": "Find employee email addresses.",
        "error_type": "sensitive_column"
    }
]

print("Total test cases:", len(evaluation_test_cases))

for test_case in evaluation_test_cases:
    print(
        test_case["id"],
        "|",
        test_case["error_type"],
        "|",
        test_case["question"]
    )

Total test cases: 8
TC01 | simple_select | Find all employee names.
TC02 | filtering | Find employees earning more than 80000.
TC03 | join | Find employees working in Engineering.
TC04 | aggregation | Find the average salary in each department.
TC05 | invalid_column | Find employee names using their name column.
TC06 | missing_join | Find employees assigned to the AI Platform project.
TC07 | incorrect_filter | Find employees working in Analytics.
TC08 | sensitive_column | Find employee email addresses.


In [66]:
evaluation_test_cases = [
    {
        "id": "TC01",
        "question": "Find all employee names.",
        "initial_sql": "SELECT employee_name FROM employees;",
        "error_type": "simple_select"
    },
    {
        "id": "TC02",
        "question": "Find employees earning more than 80000.",
        "initial_sql": "SELECT employee_name FROM employees WHERE salary > 80000;",
        "error_type": "filtering"
    },
    {
        "id": "TC03",
        "question": "Find employees working in Engineering.",
        "initial_sql": "SELECT employee_name FROM employees;",
        "error_type": "missing_join"
    },
    {
        "id": "TC04",
        "question": "Find the average salary in each department.",
        "initial_sql": """
            SELECT department_id, AVG(salary)
            FROM employees
            GROUP BY department_id;
        """,
        "error_type": "aggregation"
    },
    {
        "id": "TC05",
        "question": "Find all employee names.",
        "initial_sql": "SELECT name FROM employees;",
        "error_type": "invalid_column"
    },
    {
        "id": "TC06",
        "question": "Find employees assigned to the AI Platform project.",
        "initial_sql": """
            SELECT employee_name
            FROM employees
            WHERE project_name = 'AI Platform';
        """,
        "error_type": "invalid_column_missing_join"
    },
    {
        "id": "TC07",
        "question": "Find employees working in Analytics.",
        "initial_sql": """
            SELECT e.employee_name
            FROM employees e
            JOIN departments d
            ON e.department_id = d.department_id
            WHERE d.department_name = 'Engineering';
        """,
        "error_type": "incorrect_filter"
    },
    {
        "id": "TC08",
        "question": "Find employee email addresses.",
        "initial_sql": "SELECT email FROM employees;",
        "error_type": "sensitive_column"
    }
]

print("Benchmark test cases prepared:", len(evaluation_test_cases))

for case in evaluation_test_cases:
    print(
        case["id"],
        "|",
        case["error_type"],
        "| Initial SQL prepared"
    )

Benchmark test cases prepared: 8
TC01 | simple_select | Initial SQL prepared
TC02 | filtering | Initial SQL prepared
TC03 | missing_join | Initial SQL prepared
TC04 | aggregation | Initial SQL prepared
TC05 | invalid_column | Initial SQL prepared
TC06 | invalid_column_missing_join | Initial SQL prepared
TC07 | incorrect_filter | Initial SQL prepared
TC08 | sensitive_column | Initial SQL prepared


In [67]:
async def evaluate_sql(question: str, sql_query: str) -> EvaluationResult:
    """Evaluate SQL using deterministic checks and semantic reasoning."""

    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(VERIFIER_SERVER_PATH)]
    )

    with open(
        PROJECT_ROOT / "verifier_mcp_errors.log",
        "w",
        encoding="utf-8"
    ) as server_log:

        async with stdio_client(
            server_params,
            errlog=server_log
        ) as (read, write):

            async with ClientSession(read, write) as session:
                await session.initialize()

                tool_results = {}

                for tool_name in [
                    "check_sql_syntax",
                    "check_sql_safety",
                    "check_sql_schema",
                    "execute_sql_readonly"
                ]:
                    result = await session.call_tool(
                        tool_name,
                        {"sql_query": sql_query}
                    )

                    tool_results[tool_name] = json.loads(
                        result.content[0].text
                    )

    syntax = tool_results["check_sql_syntax"]
    safety = tool_results["check_sql_safety"]
    schema = tool_results["check_sql_schema"]
    execution = tool_results["execute_sql_readonly"]

    validation_report = {
        "syntax": syntax,
        "safety": safety,
        "schema": schema,
        "execution": {
            "valid": execution["valid"],
            "error": execution["error"]
        }
    }

    errors = []

    for name, result in [
        ("Syntax", syntax),
        ("Safety", safety),
        ("Schema", schema),
        ("Execution", execution)
    ]:
        if not result["valid"]:
            errors.append(f"{name}: {result['error']}")

    if errors:
        return EvaluationResult(
            syntax_valid=syntax["valid"],
            schema_valid=schema["valid"],
            safety_valid=safety["valid"],
            execution_success=execution["valid"],
            semantic_correct=False,
            decision="reject",
            errors=errors,
            feedback="; ".join(errors),
            confidence=1.0
        )

    return await evaluator_chain.ainvoke({
        "question": question,
        "sql_query": sql_query,
        "database_schema": evaluator_schema,
        "validation_report": json.dumps(
            validation_report,
            indent=2
        ),
        "execution_results": json.dumps(
            execution,
            indent=2,
            default=str
        ),
        "format_instructions": evaluation_parser.get_format_instructions()
    })


async def evaluate_and_repair(
    question: str,
    initial_sql: str,
    max_attempts: int = 3
) -> dict:
    """Evaluate SQL and repair eligible errors with bounded retries."""

    current_sql = initial_sql
    history = []
    stop_reason = "maximum_attempts_reached"

    for attempt in range(1, max_attempts + 1):

        print(f"\nATTEMPT {attempt}: Evaluating SQL...")

        evaluation = await evaluate_sql(question, current_sql)

        history.append({
            "attempt": attempt,
            "sql": current_sql,
            "evaluation": evaluation.model_dump()
        })

        if evaluation.decision == "accept":
            stop_reason = "accepted"
            print("SQL accepted.")
            break

        if not evaluation.safety_valid:
            stop_reason = "safety_rejection"
            print("Safety rejection. Repair stopped.")
            break

        if attempt == max_attempts:
            print("Maximum attempts reached.")
            break

        print("Sending actionable feedback to Writer Agent...")

        repair_result = await repair_sql(
            question=question,
            previous_sql=current_sql,
            evaluator_feedback=evaluation.feedback
        )

        current_sql = repair_result.sql_query

    final_evaluation = history[-1]["evaluation"]

    workflow_result = {
        "question": question,
        "initial_sql": initial_sql,
        "final_sql": current_sql,
        "final_decision": final_evaluation["decision"],
        "repair_attempts": len(history) - 1,
        "total_evaluations": len(history),
        "stop_reason": stop_reason,
        "history": history
    }

    log_evaluation(workflow_result)

    return workflow_result


print("Evaluator updated with actionable errors.")
print("Safety rejections now stop the repair loop.")
print("Controlled retry limit remains enabled.")

Evaluator updated with actionable errors.
Safety rejections now stop the repair loop.
Controlled retry limit remains enabled.


In [68]:
class EvaluationResult(BaseModel):
    syntax_valid: bool
    schema_valid: bool
    safety_valid: bool
    execution_success: bool
    semantic_correct: bool

    decision: Literal["accept", "reject"]

    errors: list[str] = Field(default_factory=list)

    feedback: str = Field(
        default="",
        description="Actionable repair instructions for the Writer Agent"
    )

    confidence: float = Field(
        default=0.5,
        ge=0.0,
        le=1.0,
        description="Evaluator confidence between 0 and 1"
    )


# Recreate parser and chain using the updated model
evaluation_parser = PydanticOutputParser(
    pydantic_object=EvaluationResult
)

evaluator_chain = evaluator_prompt | evaluator_llm | evaluation_parser

print("Updated EvaluationResult model.")
print("Feedback and confidence now have fallback defaults.")
print("Evaluator chain rebuilt successfully.")

Updated EvaluationResult model.
Feedback and confidence now have fallback defaults.
Evaluator chain rebuilt successfully.


In [69]:
def validate_evaluator_output(
    evaluation: EvaluationResult,
    validation_report: dict
) -> EvaluationResult:
    """Enforce consistency in the Evaluator's structured response."""

    deterministic_checks = [
        validation_report["syntax"]["valid"],
        validation_report["safety"]["valid"],
        validation_report["schema"]["valid"],
        validation_report["execution"]["valid"]
    ]

    all_checks_passed = all(deterministic_checks)

    # An accepted decision requires every check to pass
    if not all_checks_passed or not evaluation.semantic_correct:
        evaluation.decision = "reject"

    if not all_checks_passed:
        evaluation.semantic_correct = False

    # Never send empty feedback to the Writer
    if evaluation.decision == "reject" and not evaluation.feedback.strip():
        evaluation.feedback = (
            "Correct the following issues: "
            + "; ".join(evaluation.errors)
            if evaluation.errors
            else "Re-evaluate the SQL against the original question and database schema."
        )

    # Ensure confidence is valid
    evaluation.confidence = max(
        0.0,
        min(1.0, evaluation.confidence)
    )

    return evaluation


print("Evaluator output safeguards created.")

Evaluator output safeguards created.


In [70]:
async def evaluate_sql(question: str, sql_query: str) -> EvaluationResult:
    """Evaluate SQL with deterministic checks and guarded LLM output."""

    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(VERIFIER_SERVER_PATH)]
    )

    with open(
        PROJECT_ROOT / "verifier_mcp_errors.log",
        "w",
        encoding="utf-8"
    ) as server_log:

        async with stdio_client(
            server_params,
            errlog=server_log
        ) as (read, write):

            async with ClientSession(read, write) as session:
                await session.initialize()

                tool_results = {}

                for tool_name in [
                    "check_sql_syntax",
                    "check_sql_safety",
                    "check_sql_schema",
                    "execute_sql_readonly"
                ]:
                    result = await session.call_tool(
                        tool_name,
                        {"sql_query": sql_query}
                    )

                    tool_results[tool_name] = json.loads(
                        result.content[0].text
                    )

    syntax = tool_results["check_sql_syntax"]
    safety = tool_results["check_sql_safety"]
    schema = tool_results["check_sql_schema"]
    execution = tool_results["execute_sql_readonly"]

    validation_report = {
        "syntax": syntax,
        "safety": safety,
        "schema": schema,
        "execution": {
            "valid": execution["valid"],
            "error": execution["error"]
        }
    }

    errors = []

    for name, result in [
        ("Syntax", syntax),
        ("Safety", safety),
        ("Schema", schema),
        ("Execution", execution)
    ]:
        if not result["valid"]:
            errors.append(f"{name}: {result['error']}")

    # Deterministic failures bypass LLM evaluation
    if errors:
        return EvaluationResult(
            syntax_valid=syntax["valid"],
            schema_valid=schema["valid"],
            safety_valid=safety["valid"],
            execution_success=execution["valid"],
            semantic_correct=False,
            decision="reject",
            errors=errors,
            feedback="; ".join(errors),
            confidence=1.0
        )

    # Run schema-aware semantic evaluation
    try:
        evaluation = await evaluator_chain.ainvoke({
            "question": question,
            "sql_query": sql_query,
            "database_schema": evaluator_schema,
            "validation_report": json.dumps(
                validation_report,
                indent=2
            ),
            "execution_results": json.dumps(
                execution,
                indent=2,
                default=str
            ),
            "format_instructions": evaluation_parser.get_format_instructions()
        })

    except Exception as error:
        # Fail closed if the LLM output cannot be parsed
        return EvaluationResult(
            syntax_valid=True,
            schema_valid=True,
            safety_valid=True,
            execution_success=True,
            semantic_correct=False,
            decision="reject",
            errors=[f"LLM evaluation failed: {str(error)}"],
            feedback="Evaluator output was invalid. Retry semantic evaluation before accepting this SQL.",
            confidence=0.0
        )

    # Enforce consistency before returning the result
    evaluation = validate_evaluator_output(
        evaluation,
        validation_report
    )

    return evaluation


print("evaluate_sql() updated successfully.")
print("Deterministic checks and output safeguards connected.")

evaluate_sql() updated successfully.
Deterministic checks and output safeguards connected.


In [71]:
tc03_result = await evaluate_sql(
    question="Find employees working in Engineering.",
    sql_query="SELECT employee_name FROM employees;"
)

print(tc03_result.model_dump_json(indent=4))

{
    "syntax_valid": true,
    "schema_valid": true,
    "safety_valid": true,
    "execution_success": true,
    "semantic_correct": false,
    "decision": "reject",
    "errors": [
        "Missing filter on department_name",
        "Missing join between employees and departments"
    ],
    "feedback": "To correctly answer the question, join the employees table with the departments table on employees.department_id = departments.department_id, then filter the result with WHERE departments.department_name = 'Engineering'. The query should look like:\n\nSELECT e.employee_name\nFROM employees e\nJOIN departments d ON e.department_id = d.department_id\nWHERE d.department_name = 'Engineering';",
    "confidence": 0.95
}


In [73]:
benchmark_results = []

for case in evaluation_test_cases:

    print(f"\n{'=' * 55}")
    print(f"TEST CASE: {case['id']} | {case['error_type']}")
    print("QUESTION:", case["question"])

    try:
        result = await evaluate_and_repair(
            question=case["question"],
            initial_sql=case["initial_sql"],
            max_attempts=3
        )

        benchmark_record = {
            "test_id": case["id"],
            "error_type": case["error_type"],
            **result
        }

    except Exception as error:
        benchmark_record = {
            "test_id": case["id"],
            "error_type": case["error_type"],
            "benchmark_error": str(error)
        }

        print("TEST FAILED:", error)

    benchmark_results.append(benchmark_record)

    if "final_decision" in benchmark_record:
        print("Final decision:", benchmark_record["final_decision"])
        print("Repair attempts:", benchmark_record["repair_attempts"])

# Save fresh benchmark results, replacing the previous partial run
with open(BENCHMARK_LOG_PATH, "w", encoding="utf-8") as benchmark_file:
    for record in benchmark_results:
        benchmark_file.write(
            json.dumps(record, default=str) + "\n"
        )

print("\nBENCHMARK COMPLETED")
print("Total cases:", len(benchmark_results))
print("Saved to:", BENCHMARK_LOG_PATH)


TEST CASE: TC01 | simple_select
QUESTION: Find all employee names.

ATTEMPT 1: Evaluating SQL...
SQL accepted.
Final decision: accept
Repair attempts: 0

TEST CASE: TC02 | filtering
QUESTION: Find employees earning more than 80000.

ATTEMPT 1: Evaluating SQL...
SQL accepted.
Final decision: accept
Repair attempts: 0

TEST CASE: TC03 | missing_join
QUESTION: Find employees working in Engineering.

ATTEMPT 1: Evaluating SQL...
Sending actionable feedback to Writer Agent...

ATTEMPT 2: Evaluating SQL...
SQL accepted.
Final decision: accept
Repair attempts: 1

TEST CASE: TC04 | aggregation
QUESTION: Find the average salary in each department.

ATTEMPT 1: Evaluating SQL...
SQL accepted.
Final decision: accept
Repair attempts: 0

TEST CASE: TC05 | invalid_column
QUESTION: Find all employee names.

ATTEMPT 1: Evaluating SQL...
Sending actionable feedback to Writer Agent...

ATTEMPT 2: Evaluating SQL...
SQL accepted.
Final decision: accept
Repair attempts: 1

TEST CASE: TC06 | invalid_column_

In [74]:
successful = [
    r for r in benchmark_results
    if "final_decision" in r
]

accepted = [
    r for r in successful
    if r["final_decision"] == "accept"
]

rejected = [
    r for r in successful
    if r["final_decision"] == "reject"
]

repaired = [
    r for r in successful
    if r["repair_attempts"] > 0
]

successful_repairs = [
    r for r in repaired
    if r["final_decision"] == "accept"
]

metrics = {
    "total_test_cases": len(benchmark_results),
    "completed_cases": len(successful),
    "accepted_cases": len(accepted),
    "rejected_cases": len(rejected),
    "acceptance_rate": round(
        len(accepted) / len(successful) * 100, 2
    ) if successful else 0,
    "repair_attempts": sum(
        r["repair_attempts"] for r in successful
    ),
    "successful_repairs": len(successful_repairs),
    "repair_success_rate": round(
        len(successful_repairs) / len(repaired) * 100, 2
    ) if repaired else None,
    "average_repair_attempts": round(
        sum(r["repair_attempts"] for r in successful)
        / len(successful), 2
    ) if successful else 0
}

print(json.dumps(metrics, indent=4))

{
    "total_test_cases": 8,
    "completed_cases": 8,
    "accepted_cases": 7,
    "rejected_cases": 1,
    "acceptance_rate": 87.5,
    "repair_attempts": 4,
    "successful_repairs": 4,
    "repair_success_rate": 100.0,
    "average_repair_attempts": 0.5
}


In [75]:
for record in benchmark_results:
    print("\n" + "=" * 60)
    print("TEST:", record["test_id"])
    print("ERROR TYPE:", record["error_type"])

    if "benchmark_error" in record:
        print("ERROR:", record["benchmark_error"])
        continue

    print("QUESTION:", record["question"])
    print("INITIAL SQL:")
    print(record["initial_sql"])

    print("FINAL SQL:")
    print(record["final_sql"])

    print("DECISION:", record["final_decision"])

    print("FINAL EVALUATION:")
    print(json.dumps(
        record["history"][-1]["evaluation"],
        indent=4
    ))


TEST: TC01
ERROR TYPE: simple_select
QUESTION: Find all employee names.
INITIAL SQL:
SELECT employee_name FROM employees;
FINAL SQL:
SELECT employee_name FROM employees;
DECISION: accept
FINAL EVALUATION:
{
    "syntax_valid": true,
    "schema_valid": true,
    "safety_valid": true,
    "execution_success": true,
    "semantic_correct": true,
    "decision": "accept",
    "errors": [],
    "feedback": "",
    "confidence": 0.99
}

TEST: TC02
ERROR TYPE: filtering
QUESTION: Find employees earning more than 80000.
INITIAL SQL:
SELECT employee_name FROM employees WHERE salary > 80000;
FINAL SQL:
SELECT employee_name FROM employees WHERE salary > 80000;
DECISION: accept
FINAL EVALUATION:
{
    "syntax_valid": true,
    "schema_valid": true,
    "safety_valid": true,
    "execution_success": true,
    "semantic_correct": true,
    "decision": "accept",
    "errors": [],
    "feedback": "",
    "confidence": 0.95
}

TEST: TC03
ERROR TYPE: missing_join
QUESTION: Find employees working in En